In [2]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score, roc_auc_score
import torch.nn.functional as F


### Loading model

In [3]:
class CnnBetter(nn.Module):
    def __init__(self, num_classes: int = 10, dropout_p: float = 0.3):
        super().__init__()
        def block(in_c, out_c, drop):
            # Podwajamy kanały gdy zmniejszamy mapę o połowę — zachowujemy
            # całkowitą "pojemność informacyjną" warstwy (in_c * H * W ≈ out_c * H/2 * W/2)
            return [
                nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(),
                nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(),
                nn.MaxPool2d(2), nn.Dropout(drop),
            ]
        self.layers = nn.Sequential(
            *block(3,  32,  dropout_p),   # kanały: 3→32,   mapa: 32×32→16×16
            *block(32, 64,  dropout_p),   # kanały: 32→64,  mapa: 16×16→8×8
            *block(64, 128, dropout_p),   # kanały: 64→128, mapa: 8×8→4×4
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):         return self.layers(x)
    def predict_proba(self, x):   return torch.softmax(self(x), dim=1)
    def predict(self, x):         return torch.argmax(self.predict_proba(x), dim=1)

In [4]:
model = torch.load('CNN_ROBUST_cifar10.pth', weights_only=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CnnBetter(num_classes=10)
state_dict = torch.load('CNN_ROBUST_cifar10.pth', map_location=device)

model.load_state_dict(state_dict)
model.to(device)
model.eval()

CnnBetter(
  (layers): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.3, inplace=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (15): Dropout(p=0.3, inplace=F

Validation

In [5]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])

dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,          # pełen zbiór treningowy
    download=False,
    transform=transform
)

loader = DataLoader(dataset, batch_size=128, shuffle=False)

all_preds = [] # hard labels - numbers from 0-9 (my quasi-labels)
all_probs = [] #

model.eval()

with torch.no_grad():
    for x, _ in loader:
        x = x.to(device)

        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.append(preds.cpu())
        all_probs.append(probs.cpu())


all_preds = torch.cat(all_preds).numpy()
all_probs = torch.cat(all_probs).numpy()

np.save('victim_preds.npy', all_preds)
np.save('victim_probs.npy', all_probs)

/Users/jakubwoszczek-fullfocus/Kubiszon/Studia/Sem6/ai_safety/.venv/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [6]:
from sklearn.metrics import accuracy_score

true_labels = np.array(dataset.targets)
print("Accuracy:", accuracy_score(true_labels, all_preds))

Accuracy: 0.8428


### MAX 1000 questions

Harnessing 1000 images

In [7]:
from torch.utils.data import Subset, TensorDataset

# 1. Znajdź po 100 indeksów dla każdej z 10 klas
targets = np.array(dataset.targets)
indices_1000 = []

for i in range(10):  # CIFAR-10 ma klasy 0-9
    # Wybieramy pierwsze 100 wystąpień danej klasy
    class_indices = np.where(targets == i)[0][:100]
    indices_1000.extend(class_indices)

# 2. Stwórz podzbiór (Subset) - to są Twoje "surowe" dane wejściowe
surrogate_subset = Subset(dataset, indices_1000)

# 3. Przepuść TYLKO te 1000 zdjęć przez Victim, żeby dostać pseudoetykiety
temp_loader = DataLoader(surrogate_subset, batch_size=128, shuffle=False)

surrogate_inputs = []
surrogate_labels = []

model.eval()
with torch.no_grad():
    for x, _ in temp_loader:
        x = x.to(device)
        logits = model(x)
        # Używamy argmax (hard labels), bo zazwyczaj tak buduje się surrogate,
        # ale możesz też użyć softmax (soft labels) dla lepszej zbieżności
        preds = torch.argmax(logits, dim=1)

        surrogate_inputs.append(x.cpu())
        surrogate_labels.append(preds.cpu())

# Łączymy wszystko w jeden TensorDataset
surrogate_inputs = torch.cat(surrogate_inputs)
surrogate_labels = torch.cat(surrogate_labels)

# 4. Tworzymy docelowy DataLoader dla modelu Surrogate
surrogate_dataset = TensorDataset(surrogate_inputs, surrogate_labels)
surrogate_loader = DataLoader(surrogate_dataset, batch_size=32, shuffle=True)

print(f"Gotowe! Masz {len(surrogate_dataset)} przykładów do trenowania klona.")

Gotowe! Masz 1000 przykładów do trenowania klona.


New arch

In [8]:
class SurrogateCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SurrogateCNN, self).__init__()
        # Prostsza struktura: 3 bloki po 2 konwolucje
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 16x16

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 8x8

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 4x4
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(256 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# Inicjalizacja klona
surrogate_model = SurrogateCNN(num_classes=10).to(device)

In [9]:
import torch.optim as optim

# Ustawienia treningu
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(surrogate_model.parameters(), lr=0.001)
epochs = 20 # Przy 1000 próbkach możemy potrzebować więcej epok

surrogate_model.train()
for epoch in range(epochs):
    running_loss = 0.0
    for inputs, labels in surrogate_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = surrogate_model(inputs)
        loss = criterion(outputs, labels) # labels to odpowiedzi od Victim
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(surrogate_loader):.4f}")

print("Trening Surrogate zakończony!")

Epoch 10/20, Loss: 0.7204
Epoch 20/20, Loss: 0.0830
Trening Surrogate zakończony!


#### Eval surrogate

In [10]:
# 1. Znajdź 250 NOWYCH indeksów (25 per class), które nie są w indices_1000
indices_1000_set = set(indices_1000)
indices_test_250 = []

for i in range(10):
    # Szukamy wszystkich indeksów danej klasy
    all_class_indices = np.where(targets == i)[0]
    # Filtrujemy te, które już zużyliśmy do treningu (indices_1000)
    new_indices = [idx for idx in all_class_indices if idx not in indices_1000_set]
    # Bierzemy pierwsze 25 z pozostałych
    indices_test_250.extend(new_indices[:25])

# 2. Pobieramy dane i etykiety (Victim oraz True) dla test setu
test_subset = Subset(dataset, indices_test_250)
test_loader = DataLoader(test_subset, batch_size=128, shuffle=False)

test_inputs = []
test_victim_labels = []
test_true_labels = np.array(dataset.targets)[indices_test_250]

model.eval() # Model ofiary
with torch.no_grad():
    for x, _ in test_loader:
        x = x.to(device)
        preds = torch.argmax(model(x), dim=1)
        test_inputs.append(x.cpu())
        test_victim_labels.append(preds.cpu())

test_inputs = torch.cat(test_inputs)
test_victim_labels = torch.cat(test_victim_labels).numpy()

# 3. Ewaluacja modelu Surrogate na nowym zbiorze testowym
surrogate_model.eval()
test_surrogate_preds = []

with torch.no_grad():
    # Przepuszczamy test_inputs przez nasz wytrenowany surrogate_model
    outputs = surrogate_model(test_inputs.to(device))
    test_surrogate_preds = torch.argmax(outputs, dim=1).cpu().numpy()

# 4. Wyniki
final_acc_vs_victim = accuracy_score(test_victim_labels, test_surrogate_preds)
final_acc_vs_true = accuracy_score(test_true_labels, test_surrogate_preds)

print(f"--- TEST na 250 nowych próbkach (25 per class) ---")
print(f"Accuracy (Surrogate vs Victim): {final_acc_vs_victim:.4f}")
print(f"Accuracy (Surrogate vs True Labels): {final_acc_vs_true:.4f}")

--- TEST na 250 nowych próbkach (25 per class) ---
Accuracy (Surrogate vs Victim): 0.5000
Accuracy (Surrogate vs True Labels): 0.4360


#### PGD attack on surrogate 1000

In [11]:
import torch.nn.functional as F

def pgd_attack(model, images, labels, eps=0.03, alpha=0.01, iters=40):
    """
    Wykonuje atak PGD na zadany model.
    eps: maksymalna zmiana piksela
    alpha: wielkość kroku w każdej iteracji
    iters: liczba iteracji
    """
    images = images.to(device)
    labels = labels.to(device)

    # Oryginalne obrazy do rzutowania (clipping)
    ori_images = images.data

    # Startujemy od losowego punktu w otoczeniu eps (opcjonalne, ale zwiększa skuteczność)
    images = images + torch.empty_like(images).uniform_(-eps, eps)
    images = torch.clamp(images, 0, 1) # Zakładając normalizację do [0,1]

    for i in range(iters):
        images.requires_grad = True
        outputs = model(images)

        model.zero_grad()
        loss = F.cross_entropy(outputs, labels)
        loss.backward()

        # Gradient ascent
        adv_images = images + alpha * images.grad.sign()

        # Rzutowanie (Projection step) - upewniamy się, że nie wyjdziemy poza L_inf ball
        eta = torch.clamp(adv_images - ori_images, min=-eps, max=eps)
        images = torch.clamp(ori_images + eta, min=0, max=1).detach()

    return images

print("Funkcja PGD zdefiniowana.")

Funkcja PGD zdefiniowana.


In [12]:
# Ustawienia ataku
EPSILON = 0.031  # Przykładowa siła ataku (8/255)
ALPHA = 0.007    # Krok (2/255)
ITERS = 20

# 1. Generowanie przykładów adwersarialnych na modelu SURROGATE
surrogate_model.eval()
model.eval() # Victim

# Konwertujemy etykiety na tensor
labels_tensor = torch.LongTensor(test_victim_labels).to(device)

# Generujemy ataki
adv_images = pgd_attack(surrogate_model, test_inputs, labels_tensor, eps=EPSILON, alpha=ALPHA, iters=ITERS)

# 2. Testowanie skuteczności na modelu VICTIM
with torch.no_grad():
    # Przewidywania ofiary dla czystych obrazów (już je masz, ale dla pewności:)
    clean_outputs = model(test_inputs.to(device))
    clean_preds = torch.argmax(clean_outputs, dim=1)

    # Przewidywania ofiary dla obrazów zaatakowanych (Adversarial)
    adv_outputs = model(adv_images)
    adv_preds = torch.argmax(adv_outputs, dim=1)

# 3. Obliczanie metryk
clean_acc = accuracy_score(test_true_labels, clean_preds.cpu().numpy())
adv_acc = accuracy_score(test_true_labels, adv_preds.cpu().numpy())
attack_success_rate = (clean_preds == labels_tensor).float().mean().item() - (adv_preds == labels_tensor).float().mean().item()

print(f"--- WYNIKI ATAKU PGD (Transferability) ---")
print(f"Accuracy Victim na czystych danych: {clean_acc:.4f}")
print(f"Accuracy Victim na zaatakowanych danych: {adv_acc:.4f}")
print(f"Spadek celności (skuteczność ataku): {clean_acc - adv_acc:.4f}")

--- WYNIKI ATAKU PGD (Transferability) ---
Accuracy Victim na czystych danych: 0.8480
Accuracy Victim na zaatakowanych danych: 0.5600
Spadek celności (skuteczność ataku): 0.2880


### powtórz proces budowy surrogate, wykorzystując alternatywny zbiór danych (np. CINIC-10) zamiast oryginalnego zbioru treningowego.

In [13]:
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, TensorDataset

# Ścieżka do CINIC (używamy zbioru train lub valid dla szybkości)
cinic_path = './CINIC/train'

# Ważne: Normalizacja musi być taka sama, jakiej oczekuje model Victim!
cinic_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])

cinic_dataset = datasets.ImageFolder(root=cinic_path, transform=cinic_transform)
cinic_loader = DataLoader(cinic_dataset, batch_size=128, shuffle=True)

# --- KROK 1: Odpytanie Victim danymi z CINIC (Etykietowanie) ---
print("Generowanie pseudoetykiet z modelu Victim dla CINIC...")
cinic_inputs = []
cinic_pseudo_labels = []

model.eval() # Victim
with torch.no_grad():
    # Pobierzemy np. 10 000 obrazków z CINIC, żeby trening miał sens
    for i, (images, _) in enumerate(cinic_loader):
        if i > 80: break # ok. 10k obrazków
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        cinic_inputs.append(images.cpu())
        cinic_pseudo_labels.append(preds.cpu())

surrogate_train_ds = TensorDataset(torch.cat(cinic_inputs), torch.cat(cinic_pseudo_labels))
surrogate_train_loader = DataLoader(surrogate_train_ds, batch_size=64, shuffle=True)

Generowanie pseudoetykiet z modelu Victim dla CINIC...


Trening suroggate

In [14]:
surrogate_model = SurrogateCNN(num_classes=10).to(device)
optimizer = torch.optim.Adam(surrogate_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print("Trening modelu Surrogate na danych CINIC...")
surrogate_model.train()
for epoch in range(15):

    running_loss = 0.0
    for imgs, lbls in surrogate_train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        loss = criterion(surrogate_model(imgs), lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, loss: {running_loss/len(surrogate_train_loader):.4f}.")

Trening modelu Surrogate na danych CINIC...
Epoch 1, loss: 1.7539.
Epoch 2, loss: 1.3809.
Epoch 3, loss: 1.1903.
Epoch 4, loss: 1.0164.
Epoch 5, loss: 0.9049.
Epoch 6, loss: 0.7652.
Epoch 7, loss: 0.6520.
Epoch 8, loss: 0.5403.
Epoch 9, loss: 0.4499.
Epoch 10, loss: 0.3740.
Epoch 11, loss: 0.3005.
Epoch 12, loss: 0.2575.
Epoch 13, loss: 0.2424.
Epoch 14, loss: 0.1953.
Epoch 15, loss: 0.1731.


FGSM attack

In [20]:
def fgsm_attack(image, epsilon, data_grad):
    # Wyznaczamy znak gradientu
    sign_data_grad = data_grad.sign()
    # Tworzymy zaburzony obraz
    perturbed_image = image + epsilon * sign_data_grad
    # Przycinamy do zakresu [min, max] po normalizacji (uproszczenie: do 0,1 jeśli bez norm)
    # Ale bezpieczniej przyciąć do wartości sensownych dla tensora:
    perturbed_image = torch.clamp(perturbed_image, -3, 3)
    return perturbed_image

def perform_attack(target_model, surrogate_model, loader, epsilon):
    correct_before = 0
    correct_after = 0
    total = 0

    surrogate_model.eval()
    target_model.eval()

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        images.requires_grad = True

        # 1. Forward pass przez SURROGATE, żeby dostać gradienty
        outputs_surr = surrogate_model(images)
        loss = F.cross_entropy(outputs_surr, labels)
        surrogate_model.zero_grad()
        loss.backward()
        data_grad = images.grad.data

        # 2. Generowanie ataku
        perturbed_data = fgsm_attack(images, epsilon, data_grad)

        # 3. Sprawdzenie skuteczności na modelu VICTIM
        with torch.no_grad():
            # Przed atakiem
            out_before = target_model(images)
            pred_before = out_before.argmax(dim=1)
            correct_before += (pred_before == labels).sum().item()

            # Po ataku
            out_after = target_model(perturbed_data)
            pred_after = out_after.argmax(dim=1)
            correct_after += (pred_after == labels).sum().item()

            total += labels.size(0)

    return correct_before / total, correct_after / total

In [21]:
# Używamy np. małego zestawu z CIFAR-10 Test, żeby sprawdzić transferowalność
test_loader = DataLoader(dataset, batch_size=100, shuffle=True) # Twój oryginalny CIFAR
epsilons = [0, 0.05, 0.1, 0.2]

print(f"\n{'Epsilon':<10} | {'Acc Before':<12} | {'Acc After (Victim)':<12}")
print("-" * 45)

for eps in epsilons:
    acc_b, acc_a = perform_attack(model, surrogate_model, test_loader, eps)
    print(f"{eps:<10} | {acc_b:<12.4f} | {acc_a:<12.4f}")


Epsilon    | Acc Before   | Acc After (Victim)
---------------------------------------------
0          | 0.8428       | 0.8428      
0.05       | 0.8428       | 0.8135      
0.1        | 0.8428       | 0.7720      
0.2        | 0.8428       | 0.6656      


Obiczam gradient ataku na suroggate używając CINIC, aplikuje na cifar gradient i sprawdzam accuracy na CNN